# Welcome — Binder wrapper for `analogdevicesinc/arduino`

This notebook was launched by [mybinder.org](https://mybinder.org/) from a **thin wrapper repository** that builds a Jupyter environment around the upstream Analog Devices Arduino library (the upstream repo has no Binder config of its own).

The upstream library source has been cloned into the container at:

```
/home/jovyan/arduino-lib
```

and `arduino-cli` is installed on `PATH` so you can compile sketches inside the notebook (no real Arduino hardware is attached to this cloud container — flashing won't work, but compiling will).

In [ ]:
import os, subprocess, pathlib, textwrap

LIB = pathlib.Path('/home/jovyan/arduino-lib')
print('Library present at:', LIB, '→', 'OK' if LIB.is_dir() else 'MISSING')
print('Top-level entries:')
for p in sorted(LIB.iterdir())[:25]:
    print(' ', p.name + ('/' if p.is_dir() else ''))

## 1. `arduino-cli` version & core index

In [ ]:
print(subprocess.run(['arduino-cli', 'version'], capture_output=True, text=True).stdout)
print(subprocess.run(['arduino-cli', 'core', 'list'], capture_output=True, text=True).stdout or '(no cores installed — see postBuild log)')

## 2. Find example sketches shipped with the library

In [ ]:
examples = sorted(LIB.rglob('*.ino'))
print(f'Found {len(examples)} *.ino sketches')
for p in examples[:20]:
    print(' ', p.relative_to(LIB))

## 3. Compile one example (best-effort) for `arduino:avr:uno`

Cloud containers can't flash a real board, but `arduino-cli compile` will exercise the toolchain + library so you can see if a sketch builds.

In [ ]:
if not examples:
    raise RuntimeError('No .ino examples found — upstream layout changed?')
sketch = examples[0]
print('Compiling:', sketch.relative_to(LIB))
r = subprocess.run(['arduino-cli', 'compile', '--fqbn', 'arduino:avr:uno', str(sketch)],
                 capture_output=True, text=True)
print('rc=', r.returncode)
print(r.stdout[-2000:])
if r.returncode != 0:
    print('--- stderr ---')
    print(r.stderr[-2000:])

## 4. Where to go from here

- Browse the library source in the JupyterLab file browser on the left.
- Edit the upstream sources live. Changes **don't persist** after the session ends — use **File → Download** to keep them.
- To track a newer `master` commit, bump `ARG ARDUINO_REPO_REF=...` in the wrapper's `Dockerfile`, commit, push, and relaunch.